# Séance 7 — Choisir un seuil : coût, capacité et décision
*Session 7 — Choosing a threshold: cost, capacity and decision*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/07_seuil_economique.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ● séance complète |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ● séance complète |

**À la fin de cette séance, vous saurez :**

1. traduire une matrice de confusion en **dollars** (pertes et coûts de revue) ;
2. trouver le seuil qui minimise le coût total ;
3. intégrer une contrainte de capacité (alertes par jour) ;
4. défendre un seuil devant un comité de gestion.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

In [ ]:
cfg = preparer("entrainer")
eco = cfg["modelisation"]["economie"]
resume = lire_json(cfg.chemin("artefacts") / "modele" / "resume_v1.json")
print("Modèle retenu :", resume["modele_retenu"], "| seuil :", resume["seuil_decision"])
print("Économie :", eco)
image(cfg.chemin("artefacts") / "modele" / "11_cout_seuil.png")

In [ ]:
from fraude360.modeling import registre, donnees, evaluation
modele = registre.charger_par_alias("champion", cfg)          # le modèle servi en production
jeux = donnees.decouper(cfg)
variables = jeux.variables_numeriques + jeux.variables_categorielles
va = jeux.validation
y, montants = va["est_fraude"].values, va["montant"].values
scores = modele.predict_proba(va[variables])[:, 1]
nb_jours = va["jour_index"].nunique()
print(len(va), "transactions de validation sur", nb_jours, "jours")

| | Prédit fraude | Prédit légitime |
|---|---|---|
| **Fraude réelle** | VP : coût de traitement (4 $) | FN : on perd le **montant** |
| **Légitime** | FP : revue manuelle + friction (12 $) | VN : 0 $ |

### Exercice 7.1 — 🟢 Tous les parcours

Écrivez `cout_total(y, prediction, montants, eco)` = somme des montants des FN × `cout_faux_negatif_ratio`
+ nombre de FP × `cout_faux_positif` + nombre de VP × `cout_vrai_positif`.

In [ ]:
# VOTRE CODE ICI
def cout_total(y, prediction, montants, eco):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_ref = evaluation.cout_metier(y, (scores >= 0.5).astype(int), montants, eco)["cout_total"]
verifier_proche(cout_total(y, scores >= 0.5, montants, eco), _ref, 1e-6)

### Exercice 7.2 — 🟢 Tous les parcours

Parcourez 199 seuils (`np.linspace(0.005, 0.995, 199)`), calculez le coût de chacun et retenez `seuil_min`,
le seuil de coût minimal, et `cout_min`. Tracez la courbe coût-seuil.

In [ ]:
# VOTRE CODE ICI
seuils = np.linspace(0.005, 0.995, 199)
seuil_min, cout_min = ..., ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_s, _ = evaluation.seuil_optimal(y, scores, montants, eco)
_c = cout_total(y, scores >= _s, montants, eco)
verifier(abs(cout_min - _c) <= 0.05 * _c, f"Votre coût minimal devrait être proche de celui du pipeline ({_c:,.0f} $)")

### Exercice 7.3 — 🟣 Gestion + Informatique

L'équipe d'enquête traite au plus **20 alertes par jour**. Trouvez `seuil_capacite`, le plus **petit** seuil de la
grille tel que le nombre moyen d'alertes par jour de validation ne dépasse pas 20. Combien coûte cette contrainte par rapport au
seuil optimal ?

In [ ]:
# VOTRE CODE ICI
seuil_capacite = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier((scores >= seuil_capacite).sum() / nb_jours <= 20)
_i = list(np.round(seuils, 6)).index(round(seuil_capacite, 6))
verifier(_i == 0 or (scores >= seuils[_i - 1]).sum() / nb_jours > 20, "Prenez le plus petit seuil admissible")

**❓ Question (🔶 Gestion — décision d'affaires) —** Le coût d'une revue manuelle passe de 12 $ à 25 $ (nouvelle convention collective). Le seuil optimal monte-t-il ou descend-il ? Refaites le calcul avec `eco2 = {**eco, 'cout_faux_positif': 25.0}`.

*Votre réponse ici.*

## Bilan

- Un seuil se choisit en dollars, pas à 0,5 par défaut.
- La capacité opérationnelle est une contrainte aussi réelle que le coût.
- Documenter l'économie de la décision dans la configuration rend le seuil auditable.

**Pour aller plus loin :**

- `evaluation.seuil_optimal` gère aussi le budget d'alertes : comparez avec votre solution.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*